In [134]:
import pandas as pd

In [135]:
def oracle(df, opt):
    print('Energy overhead of the dynamic approach compared to an oracle-based approach')

    mt = df[df['threads'] == 'mt'].copy()
    df = df[df['threads'] != 'mt'].copy()

    mt['runtime'] = mt.apply(lambda row: row['runtime'] / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['runtime'] * 100 - 100, axis=1)
    mt['energy']  = mt.apply(lambda row: row['energy']  / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['energy']  * 100 - 100, axis=1)

    return mt.groupby('size').mean(numeric_only=True).round(0).astype(int)

In [146]:
def mean(df):
    print('\nEnergy overhead of the dynamic approach compared to a static approach')
    
    # Normalise runtime and energy consumption
    mt = df[df['threads'] == 'mt']
    df['runtime'] = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['runtime'] / row['runtime'] * 100, axis=1)
    df['energy']  = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['energy']  / row['energy']  * 100, axis=1)

    # Drop unused columns and rows
    df = df.drop('size', axis=1)
    df = df[df['threads'] != 'mt']
    df['threads'] = df['threads'].astype(int)

    # Mean per thread-count
    return df.groupby('threads').mean(numeric_only=True).round(0).astype(int)

In [150]:
df = pd.read_csv('data/compare_rust.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
df = df[df['threads'] != 'r']
print(df)
print()

print(oracle(df[df['pin'] == True], { 500: '16', 1000: '16', 1500: '12' }))
print(mean(df[df['pin'] == True]))

print()

#print(oracle(df[df['pin'] == False], { 500: '16', 1000: '16', 1500: '8' }))
#print(mean(df[df['pin'] == False]))

   threads  size    pin    runtime      energy
0        1   500   True   0.434418    6.952127
1        8   500   True   0.092826    2.622298
2       12   500   True   0.051007    2.119432
3       16   500   True   0.047007    2.048257
4       mt   500   True   0.049526    2.085427
6        1  1000   True   3.544349   55.664833
7        8  1000   True   0.814826   22.278765
8       12  1000   True   0.430929   17.902699
9       16  1000   True   0.414202   17.797915
10      mt  1000   True   0.445240   18.478033
12       1  1500   True  13.555772  204.213030
13       8  1500   True   4.734145  117.994050
14      12  1500   True   1.869299   74.207794
15      16  1500   True   2.298363   90.618170
16      mt  1500   True   2.087315   80.382645
18       1   500  False   0.434781    7.097137
19       8   500  False   0.062882    2.315913
20      12   500  False   0.051387    2.135437
21      16   500  False   0.047244    2.065750
22       e   500  False   0.048366    2.079857

Energy overh

/tmp/ipykernel_340544/3573593895.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['runtime'] = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['runtime'] / row['runtime'] * 100, axis=1)
/tmp/ipykernel_340544/3573593895.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['energy']  = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['energy']  / row['energy']  * 100, axis=1)


In [127]:
df = pd.read_csv('data/compare_matmul.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
print(df)
print(oracle(df, { 500: '16', 1000: '16', 1500: '1' }))
print(mean(df))

    size threads   runtime      energy
0    500       1  0.069821    1.134244
1    500       8  0.020135    0.571351
2    500      12  0.014550    0.518474
3    500      16  0.012188    0.518196
4   1000       1  0.844200   13.462192
5   1000       8  0.666753   16.997925
6   1000      12  0.424623   13.899220
7   1000      16  0.329986   13.046579
8   1500       1  3.113636   49.583771
9   1500       8  4.407424  108.947678
10  1500      12  2.955442   93.204659
11  1500      16  2.232449   84.923439
12   500      mt  0.012066    0.500491
13  1000      mt  0.362913   13.238356
14  1500      mt  2.371466   87.211227
Energy overhead of the dynamic approach compared to an oracle-based approach
      runtime  energy
size                 
500        -1      -3
1000       10       1
1500      -24      76

Energy overhead of the dynamic approach compared to a static approach
    size threads    runtime     energy
0    500       1  82.718241  55.874522
1    500       8  40.071271  12.402276
2

In [128]:
#df = pd.read_csv('data/compare_nbody.csv')
#df = df.drop(['runtimesd', 'energysd'], axis=1)
#oracle(df, { 10000: 16, 25000: 16, 40000: 16 })
#mean(df)

In [129]:
df = pd.read_csv('data/compare_relax.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
print(oracle(df, { 10000: '16', 25000: '14', 40000: '12' }))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
       runtime  energy
size                  
10000        9       8
25000       30      17
40000       31      14

Energy overhead of the dynamic approach compared to a static approach
     size threads    runtime     energy
0   10000       8  45.026103  13.437005
1   10000      12  17.741830   0.061218
2   10000      14   4.203699  -5.584899
3   10000      16  -8.814027  -8.213916
4   25000       8  24.858223   4.680328
5   25000      12 -12.506465 -12.282053
6   25000      14 -30.445454 -17.411273
7   25000      16 -24.891769  -2.597276
8   40000       8  12.154340   0.337482
9   40000      12 -30.947030 -14.001960
10  40000      14 -23.004376   0.240154
11  40000      16  -9.271221  15.281058
12  10000      mt   0.000000   0.000000
13  25000      mt   0.000000   0.000000
14  40000      mt   0.000000   0.000000
         runtime  energy
threads                 
8             27       6
12            -9      